In [2]:
SKIP_LLM_CHUNKING = True

## Ingestion


In [3]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [4]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")

FAQ documents: 1285
Evidently documents: 95


## Chunking and Preprocessing

### Intelligent Chunking with LLM

In [5]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

# now you can access them
openai_key = os.getenv("OPENAI_API_KEY")
print("Has key?", bool(openai_key))

if not SKIP_LLM_CHUNKING:
    openai_client = OpenAI(openai_key)

    def llm(prompt, model='gpt-4o-mini'):
        messages = [
            {"role": "user", "content": prompt}
        ]

        response = openai_client.responses.create(
            model='gpt-4o-mini',
            input=messages
        )

        return response.output_text


Has key? True


In [6]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


In [7]:
if not SKIP_LLM_CHUNKING:
    def intelligent_chunking(text):
        prompt = prompt_template.format(document=text)
        response = llm(prompt)
        sections = response.split('---')
        sections = [s.strip() for s in sections if s.strip()]
        return sections

In [8]:
from tqdm.auto import tqdm

if not SKIP_LLM_CHUNKING:
    evidently_chunks = []

    for doc in tqdm(evidently_docs):
        doc_copy = doc.copy()
        doc_content = doc_copy.pop('content')

        sections = intelligent_chunking(doc_content)
        for section in sections:
            section_doc = doc_copy.copy()
            section_doc['section'] = section
            evidently_chunks.append(section_doc)

### Simple Chunking

In [9]:
def sliding_window(seq, size, step):
  if size <= 0 or step <= 0:
    raise ValueError("size and step must be positive")

  n = len(seq)
  result = []

  for i in range(0, n, step):
    chunk = seq[i:i+size]
    result.append({'start': i, 'chunk': chunk})

    if i + size >= n:
      break
  
  return result

In [10]:
from tqdm.auto import tqdm

evidently_chunks = []
for doc in evidently_docs:
  doc_copy = doc.copy()
  doc_content = doc_copy.pop('content')
  chunks = sliding_window(doc_content, 2000, 1000)

  for chunk in chunks:
    chunk.update(doc_copy)

  evidently_chunks.extend(chunks)
  
print(f"Evidently chunks: {len(evidently_chunks)}")

Evidently chunks: 576


## Search

### Text search

In [11]:
from minsearch import Index

evidently_index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

evidently_index.fit(evidently_chunks)

In [12]:
query = 'What should be in a test dataset for AI evaluation?'
results = evidently_index.search(query)

print(f"Top result: {results[0]['chunk'][:500]}...")

Top result: Retrieval-Augmented Generation (RAG) systems rely on retrieving answers from a knowledge base before generating responses. To evaluate them effectively, you need a test dataset that reflects what the system *should* know.

Instead of manually creating test cases, you can generate them directly from your knowledge source, ensuring accurate and relevant ground truth data.

## Create a RAG test dataset

You can generate ground truth RAG dataset from your data source.

### 1. Create a Project

In th...


### Vector search

In [13]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [14]:
from minsearch import VectorSearch
from tqdm.auto import tqdm
import numpy as np

evidently_embeddings = []

for d in tqdm(evidently_chunks):
  v = embedding_model.encode(d['chunk'])
  evidently_embeddings.append(v)

evidently_embeddings = np.array(evidently_embeddings)

evidently_vindex = VectorSearch()
evidently_vindex.fit(evidently_embeddings, evidently_chunks)

  0%|          | 0/576 [00:00<?, ?it/s]

In [15]:
print(evidently_embeddings.shape)
print(len(evidently_chunks))

(576, 768)
576


In [16]:
query = 'What should be in a test dataset for AI evaluation?'
q = embedding_model.encode(query)
results = evidently_vindex.search(q)

print(f"Top result: {results[0]['chunk'][:500]}...")

Top result: When working on an AI system, you need test data to run automated evaluations for quality and safety. A test dataset is a structured set of test cases. It can contain:

* Just the inputs, or
* Both inputs and expected outputs (ground truth).

You can use this test dataset to:

* Run **experiments** and track if changes improve or degrade system performance.
* Run **regression testing** to ensure updates don’t break what was already working.
* **Stress-test** your system with complex or adversari...


### Hybrid search

In [17]:
def pure_text_search(query, num_results):
  return evidently_index.search(query, num_results=num_results)

def vector_search(query, num_results):
  q = embedding_model.encode(query)
  return evidently_vindex.search(q, num_results=num_results)

def hybrid_search(query, num_results):
  text_results = pure_text_search(query, num_results)
  vector_results = vector_search(query, num_results)

  # Combine and deduplicate results
  seen_ids = set()
  combined_results = []

  for result in text_results + vector_results:
    if result['filename'] not in seen_ids:
      seen_ids.add(result['filename'])
      combined_results.append(result)

  return combined_results

## Agents and Tools

In [18]:
system_prompt = """
You are a helpful assistant about Evidently - an open-source Python library to evaluate, test, and monitor ML and LLM systems, from experiments to production.

Use the search tool to find relevant information before answering questions.

If you can find specific information through search, use it to provide accurate
answers.

If the search doesn't return relevant results, let the user know and provide
general guidance
"""

In [19]:
from typing import List, Any
def text_search(query: str) -> List[Any]:
  """
  Perform a text-based search on the Evidently index.
  Args:
    query (str): The search query string.
  Returns:
    List[Any]: A list of up to 5 search results returned by the Evidently index.
  """
  return hybrid_search(query, num_results=5)

In [20]:
from pydantic_ai import Agent
from pydantic_ai import Agent

agent = Agent(
  name="evidently_agent",
  instructions=system_prompt,
  tools=[text_search],
  model='gpt-4o-mini'
)

c:\Projects\Courses\aihero-jm\.venv\Lib\site-packages\pydantic_ai\models\__init__.py:1280: DeprecationWarning: Specifying a model name without a provider prefix is deprecated. Instead of 'gpt-4o-mini', use 'openai:gpt-4o-mini'.
  provider_name, model_name = parse_model_id(model)


In [21]:
%load_ext dotenv
%dotenv .env

question = "What should be in a test dataset for AI evaluation?"
result = await agent.run(user_prompt=question)

print(result)

AgentRunResult(output="When creating a test dataset for AI evaluation, several key considerations and components should be included to ensure comprehensive assessment:\n\n1. **Diversity of Inputs**: The dataset should include a wide range of input examples that reflect real-world use cases, including typical scenarios, edge cases, and adversarial inputs. This helps in evaluating the robustness of the AI system.\n\n2. **Clear Ground Truth Labels**: Each input should have an associated ground truth label or expected output that serves as a reference for evaluating the model's performance.\n\n3. **Variability in Difficulty**: Include cases that vary in complexity to test different aspects of the model’s capabilities, such as easy, moderate, and difficult examples.\n\n4. **Synthetic Data Generation**: If real data is insufficient, consider generating synthetic data to fill gaps. This can be useful for creating structured test cases quickly and for scenarios that have not been encountered i

## Evaluation

### Logging

In [29]:
from pydantic_ai.messages import ModelMessagesTypeAdapter

def log_entry(agent, messages, source="user"):
  tools = []
  
  for ts in agent.toolsets:
    tools.extend(ts.tools.keys())

  dict_messages = ModelMessagesTypeAdapter.dump_python(messages)
  
  return {
    "agent_name": agent.name,
    "system_prompt": agent._instructions,
    "provider": agent.model.system,
    "model": agent.model.model_name,
    "tools": tools,
    "messages": dict_messages,
    "source": source
  }

In [36]:
import json
import secrets
from pathlib import Path
from datetime import datetime

LOG_DIR = Path('logs')
LOG_DIR.mkdir(exist_ok=True)

def serializer(obj):
  if isinstance(obj, datetime):
    return obj.isoformat()
  raise TypeError(f"Type {type(obj)} not serializable")

def log_interaction_to_file(agent, messages, source='user'):
  entry = log_entry(agent, messages, source)

  ts = entry['messages'][-1]['timestamp']

  if not isinstance(ts, datetime):
    ts = datetime.fromisoformat(str(ts).replace("Z", "+00:00"))

  ts_str = ts.strftime("%Y%m%d_%H%M%S")
  rand_hex = secrets.token_hex(3)

  filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
  filepath = LOG_DIR / filename
  
  with filepath.open("w", encoding="utf-8") as f_out:
    json.dump(entry, f_out, indent=2, default=serializer)

  return filepath

In [1]:
question = "What should be in a test dataset for AI evaluation?" # input()

result = await agent.run(user_prompt=question)
print(result.output)

log_interaction_to_file(agent, result.new_messages())

NameError: name 'agent' is not defined

### Adding References

In [ ]:
system_prompt = """
You are a helpful assistant about Evidently - an open-source Python library to evaluate, test, and monitor ML and LLM systems, from experiments to production.

Use the search tool to find relevant information before answering questions.

If you can find specific information through search, use it to provide accurate answers.

Always include references by citing the filename of the source material you used.

When citing the reference, replace "evidently-main" by the full path to the GitHub repository: "https://github.com/evidentlyai/evidently/blob/main/"
Format: [LINK TITLE](FULL_GITHUB_LINK)

If the search doesn't return relevant results, let the user know and provide general guidance.
""".strip()

# Create another version of agent, let's call it faq_agent_v2
agent = Agent(
  name="evidently_agent_v2",
  instructions=system_prompt,
  tools=[text_search],
  model='gpt-4o-mini'
)

In [ ]:
question = "What should be in a test dataset for AI evaluation?" # input()

result = await agent.run(user_prompt=question)
print(result.output)

log_interaction_to_file(agent, result.new_messages())